# Adapter LoRA do Zero

> Parte da série [ML Notebooks](../README.md) — por **Nandobez**.


## Intuição

LoRA congela os pesos pré-treinados $W \in \mathbb{R}^{d \times d}$ e aprende uma atualização de baixo rank $\Delta W = BA$ com $B \in \mathbb{R}^{d \times r}, A \in \mathbb{R}^{r \times d}$ e $r \ll d$. Você treina $O(rd)$ parâmetros em vez de $O(d^2)$, e guarda $A, B$ como um adapter minúsculo por tarefa.


## Formulação Matemática

$$h = Wx + \Delta W\,x = Wx + B(Ax),\quad \text{rank}(\Delta W) \leq r$$

Tipicamente inicializamos $B = 0$ para o adapter começar como no-op, e escalamos por $\alpha / r$ para manter as atualizações bem-condicionadas.


## Implementação


In [ ]:
import torch
import torch.nn as nn


In [ ]:
class LoRALinear(nn.Module):
    """Wraps a frozen `nn.Linear` and adds a trainable low-rank update."""
    def __init__(self, base: nn.Linear, r=8, alpha=16):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad = False
        in_f, out_f = base.in_features, base.out_features
        self.A = nn.Parameter(torch.randn(r, in_f) * 0.01)
        self.B = nn.Parameter(torch.zeros(out_f, r))
        self.scale = alpha / r
    def forward(self, x):
        return self.base(x) + self.scale * (x @ self.A.T) @ self.B.T

def trainable_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)


## Experimento


In [ ]:
# Wrap a 1024×1024 linear; compare param counts
base = nn.Linear(1024, 1024)
print('full Linear params:', sum(p.numel() for p in base.parameters()))
lora = LoRALinear(nn.Linear(1024, 1024), r=8)
print('LoRA trainable params:', trainable_params(lora))


In [ ]:
# Forward should match base initially (B = 0)
x = torch.randn(2, 1024)
assert torch.allclose(lora(x), lora.base(x))
print('LoRA initial output matches frozen base.')


## Discussão

- Inicializar $B = 0$ garante que o adapter começa como no-op — equivalente ao modelo base.
- $r$ troca capacidade por custo: 4–16 é comum; tarefas muito pequenas usam até $r = 2$.
- Você pode mesclar $A, B$ de volta em $W$ na inferência, custo zero em produção.


## Referências

- Repositório da série: [github.com/Nandobez/ml-notebooks](https://github.com/Nandobez/ml-notebooks)
- Autor: [Nandobez](https://github.com/Nandobez)
